In [ ]:
import os
import csv
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager

# ✅ Setup WebDriver
chrome_options = Options()
chrome_options.add_argument("--start-maximized")
chrome_options.add_argument("--disable-blink-features=AutomationControlled")

service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=chrome_options)

# ✅ Get Search Query from User
q = input("Enter the search query: ")

# ✅ Open Google Maps Directly
search_url = f"https://www.google.com/maps/search/{q.replace(' ', '+')}"
driver.get(search_url)
time.sleep(5)  # Wait for Google Maps to load

# ✅ Output File Setup
output_file = "31marchs.csv"
if not os.path.exists(output_file):
    with open(output_file, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["Business Name", "Contact Number", "Business Type", "Address", "Google Maps Link", "Website Link"])

# ✅ Scraping Businesses
count = 0
new_entries = []

while True:
    try:
        # ✅ Find Scrollable Listings Panel
        results_panel = driver.find_element(By.XPATH, "//div[@role='feed']")
        
        # ✅ Extract Business Listings
        listings = driver.find_elements(By.XPATH, "//div[contains(@class, 'Nv2PK')]")  # Business containers

        for listing in listings:
            try:
                business_name = listing.find_element(By.XPATH, ".//div[contains(@class, 'fontHeadlineSmall')]").text.strip()

                # ✅ Extract Contact Number
                try:
                    contact_number = listing.find_element(By.XPATH, ".//span[contains(@class, 'UsdlK')]").text.strip()
                except:
                    contact_number = "N/A"

                # ✅ Extract Business Type (Category)
                try:
                    business_type = listing.find_element(By.XPATH, ".//div[contains(@class, 'W4Efsd')]/span/span").text.strip()
                except:
                    business_type = "N/A"

                # ✅ Extract Address
                try:
                    address = listing.find_element(By.XPATH, ".//div[4]/div[1]/div/div/div[2]/div[4]/div[1]/span[2]/span[2]").text.strip()
                except:
                    address = "N/A"

                # ✅ Extract Business URL (Google Maps link)
                try:
                    business_link = listing.find_element(By.TAG_NAME, "a").get_attribute("href")
                except:
                    business_link = "N/A"
    
                # ✅ Extract Website Link
                try:
                    website_link = listing.find_element(By.XPATH, ".//a[contains(@aria-label, 'Visit')]").get_attribute("href")
                except:
                    website_link = "N/A"

                # ✅ Print and Save Data
                print(f"Business: {business_name} | Contact: {contact_number} | Type: {business_type} | Address: {address} | Maps Link: {business_link} | Website: {website_link}")
                new_entries.append([business_name, contact_number, business_type, address, business_link, website_link])
                count += 1
            except Exception as e:
                print(f"❌ Error extracting business: {e}")

        # ✅ Save Data to CSV
        if new_entries:
            with open(output_file, "a", newline="", encoding="utf-8") as f:
                writer = csv.writer(f)
                writer.writerows(new_entries)
            print(f"✅ {len(new_entries)} new entries saved!")
            new_entries = []

      # ✅ Scroll to Load More Results (Faster)
        driver.execute_script("arguments[0].scrollTop += 1000;", results_panel)
        time.sleep(0.5)

        # ✅ Try Clicking "Next Page" (If Available)
        try:
            next_button = driver.find_element(By.XPATH, "//button[contains(@aria-label, 'Next')]")
            driver.execute_script("arguments[0].click();", next_button)
            print("➡️ Moving to next page...")
            time.sleep(1)
        except:
            print("❌ No more pages available. Exiting.")
            break

    except Exception as e:
        
        print(f"❌ Unexpected error: {e}")
        break

# ✅ Close WebDriver
# driver.quit()


Business: Deluxe carpenter services | Contact: 081300 50779 | Type: Furniture Manufacturer | Address:  | Maps Link: https://www.google.com/maps/place/Deluxe+carpenter+services/data=!4m7!3m6!1s0x390d055880f26ee1:0x2b16360126b92cd5!8m2!3d28.6934918!4d77.1482354!16s%2Fg%2F11lgkqnkm4!19sChIJ4W7ygFgFDTkR1Sy5JgE2Fis?authuser=0&hl=en&rclk=1 | Website: N/A
Business: Sharma Furniture | Contact: 098710 81405 | Type: Furniture Manufacturer | Address: N/A | Maps Link: https://www.google.com/maps/place/Sharma+Furniture/data=!4m7!3m6!1s0x390d03b061ac6e63:0xd1263230eee139c8!8m2!3d28.6890482!4d77.1485642!16s%2Fg%2F11h0jj4tz9!19sChIJY26sYbADDTkRyDnh7jAyJtE?authuser=0&hl=en&rclk=1 | Website: N/A
Business: HAMZA ENTERPRISES-PVT.LTD CNC ROUTER DELHI,CONTRACTOR,INTERIOR DESIGNER,FLAT,CARPENTER,FURNITURE,BUILDER,HOTEL,CATERING,INDIA | Contact: 087506 15350 | Type: Interior construction contractor | Address:  | Maps Link: https://www.google.com/maps/place/HAMZA+ENTERPRISES-PVT.LTD+CNC+ROUTER+DELHI,CONTRACT

### duplicates drop

In [1]:
import pandas as pd

# ✅ Read the CSV file
d = pd.read_csv("31march.csv")

# ✅ Drop duplicate records based on "Business Name"
d.drop_duplicates(subset=['Business Name'], keep='first', inplace=True)

# ✅ Reset index in a new sequence
d.reset_index(drop=True, inplace=True)

# ✅ Print the total unique count of Business Name
unique_count = d['Business Name'].nunique()
print(f"Total unique Business Name: {unique_count}")

# ✅ Save the cleaned DataFrame back to the same file
d.to_csv("31march.csv", index=False)

# ✅ Display the cleaned DataFrame
d

Total unique Business Name: 1181


,Business Name,Contact Number,Business Type,Address,Google Maps Link,Website Link
0,GOEL FINANCE COMPANY,098111 64480,Loan agency,"3754, MAIN ROAD",https://www.google.com/maps/place/GOEL+FINANCE...,NaN
1,Cholamandalam Investment and Finance Company L...,080 4446 4939,Loan agency,,https://www.google.com/maps/place/Cholamandala...,https://www.cholamandalam.com/branch/azadpur
2,D Wealth Services Private Limited (A Delhi Wea...,099111 18103,Financial consultant,,https://www.google.com/maps/place/D+Wealth+Ser...,https://delhiwealthgroup.com/
3,Money Vikas Financial Services,098113 40586,Financial consultant,"714, AGGARWAL MILLENIUM TOWER, 1",https://www.google.com/maps/place/Money+Vikas+...,NaN
4,Shanky Finance,011 4587 2294,Loan agency,"Unit No. 1116, 11th floor, D Mall, Netaji Subh...",https://www.google.com/maps/place/Shanky+Finan...,NaN
...,...,...,...,...,...,...
1176,ADVOCATE NADEEM SAIFI - Divorce Lawyer & Crimi...,098187 86756,Divorce lawyer,"Office No. 19, Ground Floor, Aggarwal E-Mall",https://www.google.com/maps/place/ADVOCATE+NAD...,http://nadeemsaifi.in/
1177,Pulastya Legal,011 4587 0889,Legal services,9/2,https://www.google.com/maps/place/Pulastya+Leg...,http://pulastyalegal.com/
1178,FortuneLegal Advocates,098112 53529,Legal services,"Delhi High Court, Block III, 104 Lawyers Chamber",https://www.google.com/maps/place/FortuneLegal...,http://www.fortunelegal.in/
1179,Delhi Laws Firm | Service cases lawyers | Serv...,080762 44354,Law firm,"Western Wing Central, 541",https://www.google.com/maps/place/Delhi+Laws+F...,NaN


### shuffling of records


In [2]:
import pandas as pd

def shuffle_csv(input_file, output_file):
    # Read the CSV file
    df = pd.read_csv(input_file)
    
    # Shuffle the records
    df = df.sample(frac=1).reset_index(drop=True)
    
    # Save the shuffled data to a new CSV file
    df.to_csv(output_file, index=False)
    print(f"Shuffled CSV saved as {output_file}")

# Example usage
shuffle_csv("31march.csv", "31marchfiltered.csv")


Shuffled CSV saved as 31marchfiltered.csv


### website count

In [2]:
null_counts = d["Website Link"].isnull().sum()
null_counts

95

### drop the address and Google Maps link

In [4]:
import pandas as pd
df = pd.read_csv("C:/Users/91954/OneDrive/Desktop/BROZ MEDIA/api/leads generate/31marchfiltered.csv")

# Drop the 'Age' column
df = df.drop(columns=['Address'])

df.to_csv("C:/Users/91954/OneDrive/Desktop/BROZ MEDIA/api/leads generate/31marchfiltered.csv", index=False)

### now droping rows contains the marketing word

In [15]:
import pandas as pd

df = pd.read_csv("skincarebrands.csv")
# Assuming df is your DataFrame

# Drop rows where 'Business Type' contains 'Marketing'
df = df[~df['Business Type'].str.contains('Marketing', case=False, na=False)]

# Reset index after dropping rows
df = df.reset_index(drop=True)
df.to_csv("skincarebrands.csv", index=False)
# Display updated DataFrame
print(df)


                                          Business Name Contact Number  \
0                             डॉ मोनिशा कपूर एस्थेटिक्स   088608 51727   
1     RAP Beauty - Online Luxury Skin Care and Beaut...   099909 65666   
2                 Salvia Cosmeceuticals Private Limited   097172 67428   
3        Wonderskin - Korean Natural Skin Care Products  011 4020 4356   
4     Sachdeva Beauty Collection - Forever52 - Nykaa...   098102 30015   
...                                                 ...            ...   
2098   The Optical Planet - Best Optical Shop in Bhopal   088786 22343   
2099  𝐌𝐀𝐄𝐑𝐈 - Bridal Lehenga on Rent, Jewellery, Gow...   089623 89435   
2100                      Sanyasi Ayurveda, Bhopal (MP)   093007 28630   
2101  Emedicalwala | Online Pharmacy | Buy Medicines...  0755 400 3566   
2102                                           Sparkles   074155 52659   

                                          Business Type  \
0                                       Plastic surg

### in business name marketing check

In [16]:
import pandas as pd

df = pd.read_csv("skincarebrands.csv")
# Assuming df is your DataFrame

# Drop rows where 'business_name' contains 'Marketing'
df = df[~df['Business Name'].str.contains('Marketing', case=False, na=False)]

# Reset index after dropping rows
df = df.reset_index(drop=True)
df.to_csv("skincarebrands.csv", index=False)
# Display updated DataFrame
print(df)

                                          Business Name Contact Number  \
0                             डॉ मोनिशा कपूर एस्थेटिक्स   088608 51727   
1     RAP Beauty - Online Luxury Skin Care and Beaut...   099909 65666   
2                 Salvia Cosmeceuticals Private Limited   097172 67428   
3        Wonderskin - Korean Natural Skin Care Products  011 4020 4356   
4     Sachdeva Beauty Collection - Forever52 - Nykaa...   098102 30015   
...                                                 ...            ...   
2070   The Optical Planet - Best Optical Shop in Bhopal   088786 22343   
2071  𝐌𝐀𝐄𝐑𝐈 - Bridal Lehenga on Rent, Jewellery, Gow...   089623 89435   
2072                      Sanyasi Ayurveda, Bhopal (MP)   093007 28630   
2073  Emedicalwala | Online Pharmacy | Buy Medicines...  0755 400 3566   
2074                                           Sparkles   074155 52659   

                                          Business Type  \
0                                       Plastic surg